# Random Forrest - Credit Classification
### October 2026
### By Arthur Cooper
---

In [1]:
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub


In [2]:
# Download latest version
path = kagglehub.dataset_download("parisrohan/credit-score-classification")

print("Path to dataset files:", path)

Path to dataset files: /Users/arthur/.cache/kagglehub/datasets/parisrohan/credit-score-classification/versions/1


### Read the training dataset

In [3]:
# path is the kagglehub cache folder returned by dataset_download above
# low_memory=False avoids the mixed-type warning on Monthly_Balance
train_df = pd.read_csv(os.path.join(path, 'train.csv'), low_memory=False)

print(train_df.shape)
train_df.head()

(100000, 28)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


### Prepocessing
---

In [4]:
# Testing out the python pandas-dq 1.29 preprocessing and profiling library
from pandas_dq import dq_report
from pandas_dq import Fix_DQ

In [5]:
# data cleaning and preprocessing
df_cleaned = train_df.copy()

# using knowledge of previous project I am going to drop the unimportant columns for the model
df_cleaned.drop(['Name','Month','SSN','ID','Credit_Mix','Occupation', 'Annual_Income', 'Type_of_Loan'], axis=1, inplace=True)
# This time I left customer ID in the datatset to ensure customers don't appear in both train and test sets.


# converting credit age to years
df_cleaned['Credit_History_Age'] = df_cleaned['Credit_History_Age'].str.split(' ').str[0]
df_cleaned['Credit_History_Age'] = df_cleaned['Credit_History_Age'].fillna(0)
df_cleaned['Credit_History_Age'] = df_cleaned['Credit_History_Age'].astype('int64')

# converting columns to numeric
columns_convert_num = ['Age','Num_of_Loan','Num_of_Delayed_Payment','Changed_Credit_Limit','Outstanding_Debt','Amount_invested_monthly','Monthly_Balance']
df_cleaned[columns_convert_num] = df_cleaned[columns_convert_num].apply(pd.to_numeric, errors='coerce')

# Clearly string / categorical columns
string_columns = ['Payment_of_Min_Amount','Payment_Behaviour']

# convert categorical columns into dummy variables
df_cleaned = pd.get_dummies(df_cleaned, columns=string_columns, drop_first=True,dtype=int)

# Numeric columns to check for outliers
numeric_columns = [col for col in df_cleaned.select_dtypes(include=[np.number]).columns
                   if df_cleaned[col].nunique() > 2]

# replace outliers from the dataset using IQR method with Nan
for col in numeric_columns:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    df_cleaned.loc[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound), col] = np.nan

# impute missing nuemeric values with median
df_cleaned = df_cleaned.fillna(df_cleaned.median(numeric_only=True))

# drop any remaining rows with missing values
df_cleaned = df_cleaned.dropna()

# Generate a data quality report
dqr = dq_report(df_cleaned, target=None, html=False, csv_engine="pandas", verbose=1)

    All variables classified into correct types.


/var/folders/c7/s_7n84_s6g92fqcd6c0jf1nh0000gn/T/ipykernel_96140/3712029576.py:44: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  dqr = dq_report(df_cleaned, target=None, html=False, csv_engine="pandas", verbose=1)


,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Customer_ID,object,0.000000,12,,,No issue
Age,float64,0.000000,NA,14.000000,95.000000,Column has 3 outliers greater than upper bound (65.00) or lower than lower bound(1.00). Cap them or remove them.
Monthly_Inhand_Salary,float64,0.000000,NA,303.645417,15204.633333,Column has 4365 outliers greater than upper bound (10740.69) or lower than lower bound(-3577.08). Cap them or remove them.
Num_Bank_Accounts,float64,0.000000,NA,-1.000000,18.000000,Column has 1 outliers greater than upper bound (13.00) or lower than lower bound(-3.00). Cap them or remove them.
Num_Credit_Card,float64,0.000000,NA,0.000000,16.000000,Column has 5 outliers greater than upper bound (11.50) or lower than lower bound(-0.50). Cap them or remove them.
Interest_Rate,float64,0.000000,NA,1.000000,34.000000,No issue
Num_of_Loan,float64,0.000000,NA,0.000000,17.000000,Column has 1 outliers greater than upper bound (9.50) or lower than lower bound(-2.50). Cap them or remove them.
Delay_from_due_date,float64,0.000000,NA,-5.000000,67.000000,Column has 4002 outliers greater than upper bound (55.00) or lower than lower bound(-17.00). Cap them or remove them.
Num_of_Delayed_Payment,float64,0.000000,NA,-3.000000,28.000000,No issue
Changed_Credit_Limit,float64,0.000000,NA,-6.490000,36.970000,Column has 1177 outliers greater than upper bound (28.52) or lower than lower bound(-8.44). Cap them or remove them.


In [6]:
# inspect the dataset after cleaning
print(df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 26 columns):
 #   Column                                              Non-Null Count   Dtype  
---  ------                                              --------------   -----  
 0   Customer_ID                                         100000 non-null  object 
 1   Age                                                 100000 non-null  float64
 2   Monthly_Inhand_Salary                               100000 non-null  float64
 3   Num_Bank_Accounts                                   100000 non-null  float64
 4   Num_Credit_Card                                     100000 non-null  float64
 5   Interest_Rate                                       100000 non-null  float64
 6   Num_of_Loan                                         100000 non-null  float64
 7   Delay_from_due_date                                 100000 non-null  float64
 8   Num_of_Delayed_Payment                              100000 non-nu

In [7]:
# inspect the dataset after cleaning
print(df_cleaned.describe())

            Age       Monthly_Inhand_Salary  Num_Bank_Accounts  \
count  100000.000000      100000.000000        100000.000000     
mean       33.292020        4029.084964             5.362920     
std        10.356679        2961.363540             2.577378     
min        14.000000         303.645417            -1.000000     
25%        25.000000        1792.084167             3.000000     
50%        33.000000        3093.745000             5.000000     
75%        41.000000        5371.525000             7.000000     
max        95.000000       15204.633333            18.000000     

       Num_Credit_Card  Interest_Rate   Num_of_Loan   Delay_from_due_date  \
count   100000.000000   100000.000000  100000.000000     100000.000000      
mean         5.521730       14.500450       3.485660         21.068780      
std          2.046824        8.653109       2.338814         14.860104      
min          0.000000        1.000000       0.000000         -5.000000      
25%          4.00000

In [8]:
# Check the creditscore column for class imbalance
print(df_cleaned['Credit_Score'].value_counts())

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64


The classes are imbalanced, so we will use stratified sampling to split the dataset into training and testing sets. We will use 80% of the data for training and 20% for testing. The random classifier also accounts for calss imbalance.

## Train 
---

In [9]:
# Train Test Split - stratified by Credit_Score and grouped by Customer_ID to avoid data leakage

# Features and target
X = df_cleaned.drop('Credit_Score', axis=1)
y = df_cleaned['Credit_Score']

# Customer_ID was dropped from df_cleaned, but the index still lines up with train_df
groups = train_df.loc[X.index, 'Customer_ID']

# Train Test Split - stratified by Credit_Score and grouped by Customer_ID to avoid data leakage
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(X_train.shape, X_test.shape)


(80000, 25) (20000, 25)


In [10]:
print(y_train.shape, y_test.shape)

(80000,) (20000,)


In [11]:
# Train a Random Forest Classifier
# class_weight='balanced_subsample' reweights the classes in each tree's bootstrap sample to handle the imbalance
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced_subsample',
    n_jobs=-1,          # use all CPU cores
    random_state=42
)
rf.fit(X_train, y_train)

ValueError: could not convert string to float: 'CUS_0xd40'